# RelayVSR — tải weights về Drive `SD-Data`

Tải weights cho pack node [bbaudio-2025/Comfyui-Relay-VSR](https://github.com/bbaudio-2025/Comfyui-Relay-VSR) về `SD-Data/ComfyModel/relayvsr/`.

- **Bản 4×:** [kopper/RelayVSR](https://huggingface.co/kopper/RelayVSR) gồm `wan.safetensors` (~10,1 GB) và `flash.safetensors` (~128 MB), lưu vào `RelayVSR_X4/`.
- **Bản 2×:** [lightx2v/RelayVSR_X2](https://huggingface.co/lightx2v/RelayVSR_X2) gồm `wan_x2.safetensors` (~10,1 GB) và `flash_x2.safetensors` (~128 MB), lưu vào `RelayVSR_X2/`.
- Mỗi bản **phải nằm trong thư mục riêng, đúng tên file** như trên thì node mới thấy ở ô `bundle`.
- Cần `Script.ipynb` đã nối `ComfyModel/relayvsr` → `models/relayvsr`, và `Custom_ComfyNode.txt` có dòng `https://github.com/bbaudio-2025/Comfyui-Relay-VSR`.
- Hai repo HF đều public, không cần token.

In [ ]:
#@title 📂 1. Mount Drive + chọn bản tải
import os
from google.colab import drive
drive.mount('/content/drive')

SD = '/content/drive/MyDrive/SD-Data'
Ban = "4x" #@param ["4x", "2x", "cả hai"]
HF_Token = "" #@param {type:"string"}
#@markdown Repo public, không cần token. Chỉ điền nếu bị giới hạn tốc độ (hoặc đặt Colab Secret tên HF_TOKEN).

if not HF_Token:
    try:
        from google.colab import userdata
        HF_Token = userdata.get('HF_TOKEN') or ''
    except Exception:
        HF_Token = ''

# (repo, tên file, thư mục đích, dung lượng byte để kiểm)
X4 = [("kopper/RelayVSR", "wan.safetensors", "RelayVSR_X4", 10066399024),
      ("kopper/RelayVSR", "flash.safetensors", "RelayVSR_X4", 127759784)]
X2 = [("lightx2v/RelayVSR_X2", "wan_x2.safetensors", "RelayVSR_X2", 10127172392),
      ("lightx2v/RelayVSR_X2", "flash_x2.safetensors", "RelayVSR_X2", 127759464)]
FILES = X4 if Ban == "4x" else X2 if Ban == "2x" else X4 + X2

ROOT = f"{SD}/ComfyModel/relayvsr"
for _, _, d, _ in FILES:
    os.makedirs(f"{ROOT}/{d}", exist_ok=True)

need = sum(s for *_, s in FILES) / 1e9
print(f"Cần khoảng {need:.1f} GB trên Drive. Thư mục đích: {ROOT}")
print("\n--- Dung lượng Drive còn trống ---")
!df -h /content/drive | tail -1
print("\n--- Đang có trong thư mục đích ---")
!ls -lhR "{ROOT}"

In [ ]:
#@title ⬇️ 2. Tải về đĩa Colab (tải tiếp được nếu rớt)
!apt-get -qq install -y aria2 > /dev/null
H = f'--header="Authorization: Bearer {HF_Token}"' if HF_Token else ""
for repo, name, d, size in FILES:
    url = f"https://huggingface.co/{repo}/resolve/main/{name}"
    print(f"\n>>> {d}/{name}")
    !aria2c {H} --console-log-level=error --summary-interval=15 -c -x 16 -s 16 -k 4M -d /content/relayvsr/{d} -o {name} "{url}"

In [ ]:
#@title ✅ 3. Kiểm dung lượng + chép sang Drive
import shutil
ok = True
for repo, name, d, size in FILES:
    src = f"/content/relayvsr/{d}/{name}"
    got = os.path.getsize(src) if os.path.exists(src) else -1
    if got != size:
        ok = False
        print(f"❌ {d}/{name}: có {got} byte, cần {size}. Chạy lại ô 2.")
        continue
    dst = f"{ROOT}/{d}/{name}"
    print(f"Chép {d}/{name} ...")
    shutil.copyfile(src, dst)
    print(f"✔ {dst} ({os.path.getsize(dst)/1e9:.2f} GB)")
print("\nXONG, đủ file." if ok else "\nCòn file thiếu, xem dòng ❌ ở trên.")
!ls -lhR "{ROOT}"

In [ ]:
#@title 🔍 4. (Tuỳ chọn) Kiểm GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
#@markdown Bản BF16 cần khoảng 18–19 GB VRAM cho clip 768x1344 (bật `offload` trong RelayVSR Model Loader nếu GPU ~24 GB).